# Allen-Cahn — $\Delta t$ scaling of the learned potential and rollout error

We sweep the inference step size $\Delta t$ around the training value and roll out the trained `s_onsagernet` model on Allen-Cahn test trajectories. For each $\Delta t$ we record

- $V_\theta(u^t)$ along the predicted trajectory, and
- the per-step relative RMSE against the ground-truth trajectory (matched on the truth time grid),

then plot both quantities vs prediction lead time, with one curve per $\Delta t$ (light $\to$ dark = small $\to$ large $\Delta t$, `Blues` colormap).

In [ ]:
from pathlib import Path

import rootutils
import torch

ROOT = rootutils.setup_root(Path.cwd(), indicator=".project-root", pythonpath=True)

from notebooks.potential_variation_dt_scaling.helpers import (  # noqa: E402
    DATASET_CONFIG,
    load_model_for_inference,
    load_test_data,
    make_dt_factor_plan,
    plot_potential_vs_lead_time,
    plot_rel_rmse_vs_lead_time,
    run_dt_trajectory_sweep,
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"ROOT: {ROOT}")
print(f"Device: {device}")

In [ ]:
DATASET = "ac"
TRAJ_IDXS = [1, 2, 3, 4, 5]  # test trajectories to average over
# Δt = 1e-5, 3e-5, 1e-4, 3e-4, 1e-3 (=train_dt), 3e-3
DT_FACTORS = (0.01, 0.03, 0.1, 0.3, 1.0, 3.0)

cfg = DATASET_CONFIG[DATASET]
print(f"Dataset: {DATASET}  |  train_dt={cfg['train_dt']}  |  T_final={cfg['T_final']}  |  trajs={TRAJ_IDXS}")

In [ ]:
run_dir = ROOT / "logs/official/runs" / DATASET / "s_onsagernet"
print(f"Loading s_onsagernet model for '{DATASET}' ...")
model = load_model_for_inference(run_dir, root=ROOT, device=device)
potential = model.dynamics.potential
print(f"Potential type: {type(potential).__name__}")

data_glob = str(ROOT / "data" / cfg["data_subdir"] / "*.hdf5")
test_data, t_coord, x_coord = load_test_data(data_glob)
N_test, T_data, n_vars, Nx = test_data.shape
print(f"Test set: {N_test} trajectories  |  T_data={T_data}, n_vars={n_vars}, Nx={Nx}")
dt_truth = float(t_coord[1] - t_coord[0])
print(f"dt_truth (data spacing): {dt_truth:.4e}")

In [ ]:
plan = make_dt_factor_plan(cfg["train_dt"], cfg["T_final"], factors=DT_FACTORS)

print(f"T_final = {cfg['T_final']}")
print(f"{'\u0394t':>12}  {'n_steps':>8}")
for dt, n in plan:
    print(f"{dt:>12.2e}  {n:>8d}")

sweep = run_dt_trajectory_sweep(
    model,
    test_data,
    traj_idxs=TRAJ_IDXS,
    plan=plan,
    dt_truth=dt_truth,
    device=device,
)

In [ ]:
plot_potential_vs_lead_time(
    sweep,
    train_dt=cfg["train_dt"],
    display_name=cfg["display"],
    out_path=ROOT / "figs/potential_variation_dt_scaling/ac_potential_vs_lead_time.pdf",
)

In [ ]:
plot_rel_rmse_vs_lead_time(
    sweep,
    train_dt=cfg["train_dt"],
    display_name=cfg["display"],
    out_path=ROOT / "figs/potential_variation_dt_scaling/ac_rel_rmse_vs_lead_time.pdf",
)